In [1]:
import torch
from torch import nn
class para(nn.Module):
    def __init__(self,net1,net2):
        super().__init__()
        self.net1=net1
        self.net2=net2
    def forward(self,x):
        Y1=self.net1(x)
        Y2=self.net2(x)
        return torch.cat((Y1,Y2),dim=-1)

In [2]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net1=nn.Linear(20,256)
        self.net2=nn.ReLU()
        self.net3=nn.Linear(256,20)
    def forward(self,x):
        return self.net3(self.net2(self.net1(x)))

In [3]:
def make_block():
    return nn.Sequential(
        nn.Linear(20, 20),
        nn.ReLU()
    )

In [4]:
class mynet1(nn.Module):
    def __init__(self,make_block,num):
        super().__init__()
        block=[]
        for i in range(num):
            block1=make_block()
            block.append(block1)
        net1=nn.Sequential(*block)
        net2=nn.LazyLinear(256)
        self.net1=nn.Sequential(net1,net2)
    def forward(self,x):
        return self.net1(x)
net=mynet1(make_block,2)
x=torch.rand(size=(20,20))
print(net)
y=net(x)
print(net)
print(net.net1[0][0][0].weight.data)

mynet1(
  (net1): Sequential(
    (0): Sequential(
      (0): Sequential(
        (0): Linear(in_features=20, out_features=20, bias=True)
        (1): ReLU()
      )
      (1): Sequential(
        (0): Linear(in_features=20, out_features=20, bias=True)
        (1): ReLU()
      )
    )
    (1): LazyLinear(in_features=0, out_features=256, bias=True)
  )
)
mynet1(
  (net1): Sequential(
    (0): Sequential(
      (0): Sequential(
        (0): Linear(in_features=20, out_features=20, bias=True)
        (1): ReLU()
      )
      (1): Sequential(
        (0): Linear(in_features=20, out_features=20, bias=True)
        (1): ReLU()
      )
    )
    (1): Linear(in_features=20, out_features=256, bias=True)
  )
)
tensor([[ 1.5203e-01, -1.5743e-01,  3.1410e-02,  1.4146e-01, -2.0669e-01,
          4.2756e-02, -3.2268e-02,  1.2747e-01,  9.6331e-02, -3.6483e-02,
          7.1358e-02,  1.4389e-01,  5.4105e-02, -8.8785e-02,  2.0063e-01,
          2.1409e-01, -1.0832e-01,  9.4074e-02,  1.3150e-01,  4.768

d:\miniconda\envs\d2l\lib\site-packages\torch\nn\modules\lazy.py:178: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


In [5]:
class mynet2(nn.Module):
    def __init__(self,make_block,num):
        super().__init__()
        block=[]
        for i in range(num):
            block1=make_block()
            block.append(block1)
        self.net1=nn.Sequential(*block)
        self.net2=nn.LazyLinear(256)
        #self.net1=nn.Sequential(net1,net2)
    def forward(self,x):
        for layer in self.children(): ##按照登记的顺序遍历
            x=layer(x)
        return x
net=mynet2(make_block,2)
x=torch.rand(size=(20,20))
print(net)
y=net(x)
print(net)


mynet2(
  (net1): Sequential(
    (0): Sequential(
      (0): Linear(in_features=20, out_features=20, bias=True)
      (1): ReLU()
    )
    (1): Sequential(
      (0): Linear(in_features=20, out_features=20, bias=True)
      (1): ReLU()
    )
  )
  (net2): LazyLinear(in_features=0, out_features=256, bias=True)
)
mynet2(
  (net1): Sequential(
    (0): Sequential(
      (0): Linear(in_features=20, out_features=20, bias=True)
      (1): ReLU()
    )
    (1): Sequential(
      (0): Linear(in_features=20, out_features=20, bias=True)
      (1): ReLU()
    )
  )
  (net2): Linear(in_features=20, out_features=256, bias=True)
)


In [6]:
def block1():
    return nn.Sequential(nn.Linear(20,20))
def block2():
    net=nn.Sequential()
    for i in range(4):
        net.add_module(f"{i}",block1())  #也可以net.append(nn.Linear(2,2))
    return net
net=nn.Sequential(block2(),nn.LazyLinear(10))
print(net)
print(net[1].weight)
x=torch.rand(size=(20,20))
y=net(x)
print(net[1].weight)

Sequential(
  (0): Sequential(
    (0): Sequential(
      (0): Linear(in_features=20, out_features=20, bias=True)
    )
    (1): Sequential(
      (0): Linear(in_features=20, out_features=20, bias=True)
    )
    (2): Sequential(
      (0): Linear(in_features=20, out_features=20, bias=True)
    )
    (3): Sequential(
      (0): Linear(in_features=20, out_features=20, bias=True)
    )
  )
  (1): LazyLinear(in_features=0, out_features=10, bias=True)
)
<UninitializedParameter>
Parameter containing:
tensor([[-0.0733, -0.2190, -0.1859,  0.1111, -0.1332, -0.1730,  0.0155,  0.1851,
          0.0011,  0.1007, -0.0017, -0.1699, -0.0252,  0.0933,  0.2154,  0.2035,
          0.1747,  0.0421,  0.0895, -0.2157],
        [-0.1837,  0.1201, -0.0347,  0.1101, -0.1034,  0.0460,  0.0194, -0.1685,
         -0.0433, -0.0331, -0.1680, -0.1960,  0.0043, -0.0895,  0.0067,  0.0597,
         -0.1120, -0.0654,  0.0220,  0.2215],
        [ 0.0182,  0.0807,  0.0215,  0.1237, -0.0020,  0.1800, -0.0054, -0.1026,
  

In [7]:
import torch
from torch import nn
from d2l import torch as d2l


def conv_block(input_channels, num_channels):
    return nn.Sequential(
        nn.BatchNorm2d(input_channels), nn.ReLU(),
        nn.Conv2d(input_channels, num_channels, kernel_size=3, padding=1))



In [8]:
class DenseBlock(nn.Module):
    def __init__(self, num_convs, input_channels, num_channels):
        super(DenseBlock, self).__init__()
        layer = []
        for i in range(num_convs):
            layer.append(conv_block(
                num_channels * i + input_channels, num_channels))
        self.net = nn.Sequential(*layer)

    def forward(self, X):
        for blk in self.net:
            Y = blk(X)
            # 连接通道维度上每个块的输入和输出
            X = torch.cat((X, Y), dim=1)
        return X

In [ ]:
class diynet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net1=nn.Linear(3,3)
    def forward(self,x):
        return self.net1(x)
net=nn.Sequential()
net.net1=diynet()
net.net2=nn.Linear(3,3)
print(net)

Sequential(
  (net1): diynet(
    (net1): Linear(in_features=3, out_features=3, bias=True)
  )
  (net2): Linear(in_features=3, out_features=3, bias=True)
)


In [11]:
net=diynet()
net.net2=nn.Linear(3,3)
print(net)

diynet(
  (net1): Linear(in_features=3, out_features=3, bias=True)
  (net2): Linear(in_features=3, out_features=3, bias=True)
)


# 自定义多个网络

方法一：开个空列表来接收for循环产生的网络，然后sequential绑起来

方法二：开一个空的sequential 用add_module来添加网络

方法三：也可以开一个空sequential  net.append(nn.Linear(2,2))

方法四：可以开一个空的sequential。然后用net.mynet=nn.Linear(3,3) 来自己加一层网络

方法五：使用nn.ModuleList
self.net=nn.ModuleList([transformerblock(....) for _ in range(num_layers)])
但是需要自己写循环遍历
# 自定义forward

方法一 按顺序存入sequential，然后直接访问sequential即可

方法二 for layer in self.children(): ##按照登记的顺序遍历

方法三 若一个net=sequential 也可以for layer in self.net: ##这个是针对访问sequential里层的过程 #若sequential里面还有sequential则不会再分了

方法四：for module in net.modules(): #在class外的访问方式 #class内部，for module in self.net.modules(): ！！注意方法四不可以用来前向传播，因为第一次已经执行了完整网络，后面又会重复执行内部的卷积、BN等层，容易产生维度错误.和net.apply()的遍历过程相似 只不过这个是递归遍历
children()
    只看当前模块的下一层
    可能返回Sequential、BasicBlock等整体模块

modules()
    递归展开整个网络
    既返回Sequential等容器
    也返回Conv2d、BatchNorm2d等底层模块